In [1]:
import tensorflow as tf
from tensorflow.keras.layers.experimental import preprocessing
import keras_tuner as kt

In [2]:
INPUT_SHAPE = (224, 224, 3)
DATA_DIR = './datasets/dmd/labels'
BATCH_SIZE = 32
IMG_SIZE = (180, 180)
VALIDATION_SPLIT = 0.2

In [3]:
(train_ds, val_ds) = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='int',
    validation_split=VALIDATION_SPLIT,
    subset='both',
    color_mode='grayscale',
    seed=2,
    shuffle=0,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE)

Found 26426 files belonging to 7 classes.
Using 21141 files for training.
Using 5285 files for validation.


In [4]:
class_names = train_ds.class_names
num_classes = len(class_names)
print(class_names)

['blinks_blinking', 'eyes_state_close', 'eyes_state_closing', 'eyes_state_open', 'eyes_state_opening', 'eyes_state_undefined', 'yawn']


In [5]:
layers_data_augmentation = tf.keras.Sequential([
        preprocessing.RandomFlip("horizontal"),
        preprocessing.RandomRotation(0.3, fill_mode='constant'),
        preprocessing.RandomZoom(0.2),
        tf.keras.layers.RandomBrightness(factor=0.2)]
)

In [6]:
def model_builder(hp):
    model = tf.keras.Sequential();
    # model.add(layers_data_augmentation)
    model.add(tf.keras.layers.Flatten(input_shape=IMG_SIZE))

    hp_activation = hp.Choice('activation', values=['relu', 'tanh'])
    hp_layer_1_nodes = hp.Int('layer_1_nodes', min_value=32, max_value=128, step=16)
    hp_layer_2_nodes = hp.Int('layer_2_nodes', min_value=32, max_value=128, step=16)
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.add(tf.keras.layers.Dense(units=hp_layer_1_nodes, activation=hp_activation))
    model.add(tf.keras.layers.Dense(units=hp_layer_2_nodes, activation=hp_activation))
    model.add(tf.keras.layers.Dense(num_classes, activation='softmax'))

    model.compile(
        # optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=hp_learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model

In [7]:
tuner = kt.Hyperband(model_builder,
     objective='accuracy',  # duda: deberíamos de utilizar 'val_accuracy' (validación) pero no funciona.
     max_epochs=10,
     factor=3,
     directory='hp',
     project_name='cnn_model'
     )

INFO:tensorflow:Reloading Tuner from hp\cnn_model\tuner0.json


In [8]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3)
tuner.search(train_ds, epochs=10, callbacks=[early_stop])

Trial 30 Complete [00h 00m 50s]
accuracy: 0.9689702391624451

Best accuracy So Far: 0.9825930595397949
Total elapsed time: 00h 03m 51s
INFO:tensorflow:Oracle triggered exit


In [9]:
best_hps= tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.get_best_models(num_models=1)[0]
loss, accuracy = best_model.evaluate(val_ds)
print("Accuracy: {:.2f}%".format(accuracy * 100))

166/166 [==============================] - 2s 11ms/step - loss: 1201006.6250 - accuracy: 0.3686
Accuracy: 36.86%


In [15]:
best_hp = tuner.get_best_hyperparameters()[0]
model = tuner.hypermodel.build(best_hp)

In [16]:
best_model_update = tf.keras.callbacks.ModelCheckpoint('./models/cnn/best', save_best_only=True, monitor='accuracy')
model.fit(train_ds, batch_size=BATCH_SIZE, epochs=50,  callbacks=[early_stop, best_model_update])

Epoch 1/50
661/661 [==============================] - 8s 12ms/step - loss: 19928.6445 - accuracy: 0.9796
Epoch 2/50
661/661 [==============================] - 7s 11ms/step - loss: 15844.4971 - accuracy: 0.8660
Epoch 3/50
661/661 [==============================] - 7s 11ms/step - loss: 1.2286 - accuracy: 0.6678
Epoch 4/50
661/661 [==============================] - 8s 12ms/step - loss: 1.5277 - accuracy: 0.4397
Epoch 5/50
661/661 [==============================] - 7s 11ms/step - loss: 1.4681 - accuracy: 0.4488
Epoch 6/50
661/661 [==============================] - 7s 11ms/step - loss: 1.4714 - accuracy: 0.4412


In [17]:
print(model.summary())

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_2 (Flatten)         (None, 32400)             0         
                                                                 
 dense_6 (Dense)             (None, 112)               3628912   
                                                                 
 dense_7 (Dense)             (None, 64)                7232      
                                                                 
 dense_8 (Dense)             (None, 7)                 455       
                                                                 
Total params: 3,636,599
Trainable params: 3,636,599
Non-trainable params: 0
_________________________________________________________________
None
